In [1]:
!pip install -q openai

In [2]:
!pip install -q transformers

In [3]:
from transformers import pipeline

classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [4]:
import pandas as pd

tickets = [
    "My internet connection keeps dropping every few minutes.",
    "I was charged twice for my monthly subscription.",
    "The app crashes whenever I try to upload a photo.",
    "I forgot my password and can't log into my account.",
    "The delivery for my order is 5 days late.",
    "I want to cancel my subscription and get a refund.",
    "The website shows an error when I try to checkout.",
    "My package arrived damaged and items were missing.",
    "I can't reset my password, the link isn't working.",
    "The app is very slow and takes forever to load."
]

df = pd.DataFrame({"ticket_text": tickets})
df.head(10)

,ticket_text
0,My internet connection keeps dropping every fe...
1,I was charged twice for my monthly subscription.
2,The app crashes whenever I try to upload a photo.
3,I forgot my password and can't log into my acc...
4,The delivery for my order is 5 days late.
5,I want to cancel my subscription and get a ref...
6,The website shows an error when I try to check...
7,My package arrived damaged and items were miss...
8,"I can't reset my password, the link isn't work..."
9,The app is very slow and takes forever to load.


In [5]:
candidate_labels = [
    "Billing Issue",
    "Technical Issue",
    "Account Access",
    "Delivery Problem",
    "Cancellation Request"
]

In [6]:
result = classifier(df['ticket_text'][0], candidate_labels)
print(result)

{'sequence': 'My internet connection keeps dropping every few minutes.', 'labels': ['Technical Issue', 'Delivery Problem', 'Account Access', 'Cancellation Request', 'Billing Issue'], 'scores': [0.6010789275169373, 0.15774862468242645, 0.11411844938993454, 0.0739070326089859, 0.053146932274103165]}


In [7]:
def get_top3_tags(text):
    result = classifier(text, candidate_labels)
    top3 = list(zip(result['labels'][:3], result['scores'][:3]))
    return top3

df['top3_tags'] = df['ticket_text'].apply(get_top3_tags)

for i, row in df.iterrows():
    print(f"Ticket: {row['ticket_text']}")
    print(f"Top 3 tags: {row['top3_tags']}")
    print("-" * 50)

Ticket: My internet connection keeps dropping every few minutes.
Top 3 tags: [('Technical Issue', 0.6010789275169373), ('Delivery Problem', 0.15774862468242645), ('Account Access', 0.11411844938993454)]
--------------------------------------------------
Ticket: I was charged twice for my monthly subscription.
Top 3 tags: [('Billing Issue', 0.5229536890983582), ('Account Access', 0.16238227486610413), ('Delivery Problem', 0.14842914044857025)]
--------------------------------------------------
Ticket: The app crashes whenever I try to upload a photo.
Top 3 tags: [('Technical Issue', 0.5421557426452637), ('Cancellation Request', 0.16887883841991425), ('Delivery Problem', 0.16391213238239288)]
--------------------------------------------------
Ticket: I forgot my password and can't log into my account.
Top 3 tags: [('Account Access', 0.5328936576843262), ('Technical Issue', 0.21425104141235352), ('Delivery Problem', 0.0925019159913063)]
--------------------------------------------------
T

In [8]:
few_shot_examples = """
Examples:
Ticket: "I can't log into my account, it says wrong password."
Tag: Account Access

Ticket: "I was billed twice this month for the same plan."
Tag: Billing Issue

Ticket: "My order hasn't arrived yet and it's been a week."
Tag: Delivery Problem

Now classify this new ticket:
Ticket: "{}"
"""

def few_shot_classify(text):
    prompt = few_shot_examples.format(text)
    result = classifier(prompt, candidate_labels)
    return result['labels'][0], result['scores'][0]

# Test on one ticket
tag, score = few_shot_classify(df['ticket_text'][3])
print(f"Ticket: {df['ticket_text'][3]}")
print(f"Few-shot predicted tag: {tag} (score: {score:.2f})")

Ticket: I forgot my password and can't log into my account.
Few-shot predicted tag: Account Access (score: 0.41)


In [9]:
comparison = []

for text in df['ticket_text']:
    zero_shot_result = classifier(text, candidate_labels)
    zero_tag = zero_shot_result['labels'][0]
    zero_score = zero_shot_result['scores'][0]

    few_tag, few_score = few_shot_classify(text)

    comparison.append({
        "ticket": text,
        "zero_shot_tag": zero_tag,
        "zero_shot_score": round(zero_score, 2),
        "few_shot_tag": few_tag,
        "few_shot_score": round(few_score, 2)
    })

comparison_df = pd.DataFrame(comparison)
comparison_df

,ticket,zero_shot_tag,zero_shot_score,few_shot_tag,few_shot_score
0,My internet connection keeps dropping every fe...,Technical Issue,0.60,Delivery Problem,0.52
1,I was charged twice for my monthly subscription.,Billing Issue,0.52,Billing Issue,0.46
2,The app crashes whenever I try to upload a photo.,Technical Issue,0.54,Delivery Problem,0.53
3,I forgot my password and can't log into my acc...,Account Access,0.53,Account Access,0.41
4,The delivery for my order is 5 days late.,Delivery Problem,0.74,Delivery Problem,0.63
5,I want to cancel my subscription and get a ref...,Cancellation Request,0.59,Delivery Problem,0.43
6,The website shows an error when I try to check...,Technical Issue,0.37,Delivery Problem,0.48
7,My package arrived damaged and items were miss...,Delivery Problem,0.73,Delivery Problem,0.51
8,"I can't reset my password, the link isn't work...",Technical Issue,0.60,Delivery Problem,0.40
9,The app is very slow and takes forever to load.,Technical Issue,0.63,Delivery Problem,0.50
